# OpTC Host ML — XGBoost, undersampled negatives (shared shards, GPU T4x2)

This notebook reads the same persisted 75/10/15 split and vocabulary as the other models.
Validation and test retain every event, exactly like the other notebooks.

Because the positive label is extremely rare (201,551 of 3,476,277,622 train rows, ~0.0058%), this notebook
streams the full train split exactly once with random negative undersampling (NEG_PER_POS = 30):
every attack positive row is kept, while negatives are kept with a uniform probability.
This aligns the training distribution directly with `host_rf_shards` and `host_lgbm_shards`.

Configured for Kaggle GPU environments (e.g. **GPU T4 x2** or **GPU P100**). Training and inference leverage
`tree_method='hist'` and `device='cuda'` for massive speedup over 4-core CPU.

Upload the archive created by package_host_shards.py as one dataset, then attach it.
Setup locates its manifest and maps all packaged volumes automatically. Select Accelerator: **GPU T4 x2** in Kaggle Settings.

In [ ]:
from pathlib import Path
import sys, json
# Set an explicit path here only if several Host input datasets are attached.
MANIFEST = Path(
    '/kaggle/input/datasets/nhatnguyen2101/'
    'optc-host-ml-full-input/manifest.json'
)
if MANIFEST is None:
    candidates = []
    for path in Path('/kaggle/input').rglob('manifest.json'):
        try:
            meta = json.loads(path.read_text())
        except (OSError, ValueError):
            continue
        if isinstance(meta, dict) and meta.get('format') == 'host_binary_delta_npz':
            candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Host dataset; set MANIFEST explicitly. Found: {candidates}')
    MANIFEST = candidates[0]
MANIFEST = Path(MANIFEST)
meta = json.loads(MANIFEST.read_text())
VOLUME_ROOTS = {key: MANIFEST.parent / 'volumes' / key for key in meta['volumes']}
sys.path.insert(0, str(MANIFEST.parent))
from host_shards import HostShardDataset
from shard_metrics import evaluate_stream
data = HostShardDataset(MANIFEST, VOLUME_ROOTS)
print('Manifest:', MANIFEST)
# Decode one shard per split to check the uploaded paths and format.
for split in ('train', 'val', 'test'):
    X_check, y_check = next(data.iter_batches(split))
    print(split, 'first shard:', X_check.shape, 'positive labels:', int(y_check.sum()))
    del X_check, y_check
print(json.dumps(data.meta['splits'], indent=2))
print('Full train CSR lower bound GiB:', data.csr_bytes('train') / 2**30)
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(exist_ok=True)


In [ ]:
MODEL_NAME = 'xgb'
import numpy as np
from scipy import sparse
import xgboost as xgb
import torch

# --- Detect CUDA / GPU Environment (Kaggle GPU T4 x2) ---
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
    print(f'Detected {gpu_count} GPU(s): {", ".join(gpu_names)}')
    device = 'cuda'
else:
    print('No GPU detected; falling back to CPU.')
    device = 'cpu'

# Every positive row is kept; negatives are kept with probability `keep_p`,
# streamed shard-by-shard so peak memory stays at one shard, never the full
# 3.48B-row split. This matches the sampling strategy of host_rf_shards and host_lgbm_shards.
NEG_PER_POS = 30          # negatives kept per positive, on expectation
SEED = 1
rng = np.random.default_rng(SEED)

label0 = data.meta['splits']['train']['label_0']
label1 = data.meta['splits']['train']['label_1']
keep_p = min(1.0, NEG_PER_POS * label1 / label0)
print(f'target negative keep-probability: {keep_p:.6f}')

X_parts, y_parts = [], []
n_pos = n_neg_seen = n_neg_kept = 0
for X_shard, y_shard in data.iter_batches('train'):
    pos = y_shard == 1
    if pos.any():
        X_parts.append(X_shard[pos]); y_parts.append(y_shard[pos])
        n_pos += int(pos.sum())
    neg_idx = np.flatnonzero(~pos)
    n_neg_seen += neg_idx.size
    if neg_idx.size:
        keep = neg_idx[rng.random(neg_idx.size) < keep_p]
        if keep.size:
            X_parts.append(X_shard[keep]); y_parts.append(y_shard[keep])
            n_neg_kept += keep.size

X_train = sparse.vstack(X_parts, format='csr')
y_train = np.concatenate(y_parts)
del X_parts, y_parts
print(f'kept {n_pos} positives, {n_neg_kept}/{n_neg_seen} negatives -> X_train {X_train.shape}')

# Balanced loss reweighting matching LightGBM is_unbalance
scale_pos_weight = float(n_neg_kept / max(n_pos, 1))
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

try:
    dtrain = xgb.QuantileDMatrix(X_train, label=y_train, max_bin=256)
except Exception:
    dtrain = xgb.DMatrix(X_train, label=y_train)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'tree_method': 'hist',
    'device': device,
    'max_depth': 8,
    'learning_rate': 0.05,
    'scale_pos_weight': scale_pos_weight,
    'seed': SEED,
    'verbosity': 1,
}

print(f'Training XGBoost with tree_method=hist on {device}...')
model = xgb.train(params, dtrain, num_boost_round=300)
del X_train, y_train, dtrain

# GPU-accelerated in-place prediction for evaluation
model.set_param({'device': device})
predict = lambda X: np.clip(
    np.asarray(model.inplace_predict(X, predict_type='value')).reshape(-1), 0.0, 1.0
)
model.save_model(str(OUTPUT / 'xgb_model.ubj'))
print(f'Model saved to {OUTPUT / "xgb_model.ubj"}')


In [ ]:
print('Evaluating on validation stream...')
validation = evaluate_stream(data, 'val', predict)
print(f'Validation complete: threshold={validation["threshold"]:.4f}, F1={validation["f1"]:.4f}')

print('Evaluating on test stream...')
test = evaluate_stream(data, 'test', predict, threshold=validation['threshold'])
print(f'Test complete: F1={test["f1"]:.4f}, Precision={test["precision"]:.4f}, Recall={test["recall"]:.4f}')

report = {'model': MODEL_NAME, 'split_sha256': data.meta['split_sha256'],
    'vocabulary_sha256': data.meta['vocabulary_sha256'], 'seed': SEED,
    'device': device,
    'train_sampling': {'method': 'random_negative_undersampling',
        'neg_per_pos_target': NEG_PER_POS, 'positives_kept': n_pos,
        'negatives_kept': n_neg_kept, 'negatives_seen': n_neg_seen,
        'rows_kept': n_pos + n_neg_kept},
    'validation': validation, 'test': test}
(OUTPUT / (MODEL_NAME + '_metrics.json')).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
